# 07 · Run records, replay and trace

**Role of this layer.** Every run is saved as `runs/<run_id>/manifest.json` plus
one output file per step. The manifest records the request, what the LLM
proposed and what the user changed and approved, the exact data versions (file
hashes), library versions and git commit, the registry version, every step's
output hash, the narrative and its grounding check, and stage timings.

* **replay** re-runs the final plan with the recorded approvals and *no LLM*,
  after checking the data files still match, and compares every output hash.
* **trace** answers "where does this number come from?": it walks the plan
  backwards from a step to the source files.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.lineage import load_run, replay, trace
from quantlab.mock_scenarios import SCENARIOS_BY_ID

scenario = SCENARIOS_BY_ID["buy_the_dip_tech"]  # scenario 5, the NASDAQ alone
session = run_request(scenario.request, list(scenario.dataset_ids), "mock", approve=True)
record = load_run(session.run_id, settings.runs_dir)
print("run:", record.run_id, "| status:", record.status)
print("manifest (in the runs folder):", session.manifest_path.relative_to(settings.runs_dir))
print("registry version:", record.registry_version[:16])
print("environment:", record.environment["python"], record.environment["packages"])
pd.DataFrame([d.model_dump() for d in record.datasets])

run: 20261019-091401-25732e | status: completed
manifest (in the runs folder): 20261019-091401-25732e/manifest.json
registry version: 5d0716063dd09b27
environment: 3.12.14 {'pandas': '3.0.6', 'numpy': '2.5.4', 'statsmodels': '0.15.0', 'pydantic': '2.14.1', 'openai': '3.31.0', 'anthropic': '1.14.0'}


,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,NASDAQCOM,NASDAQCOM,b1c65ade24861ff5dc51bd5bbd4d8b7ed8e68af20a6181b2457c6bfe64289520,2026-10-17T09:22:36Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=NASDAQCOM,14042,1971-02-05,2026-10-15


The saved output file of each step *is* the canonical text that was hashed, so
anyone can verify an output against the manifest with a plain SHA-256:

In [3]:
import hashlib

outputs = settings.runs_dir / record.run_id / "outputs"
for step in record.steps:
    path = next(outputs.glob(f"{step.step_id}_*"))
    same = hashlib.sha256(path.read_bytes()).hexdigest() == step.output_hash
    print(f"{path.name:28} matches manifest: {same}")

s1_load_series.csv           matches manifest: True
s2_resample.csv              matches manifest: True
s3_compute_returns.csv       matches manifest: True
s4_flag_regime.csv           matches manifest: True
s5_forward_return.csv        matches manifest: True
s6_compare_groups.json       matches manifest: True
s7_hac_mean_difference.json  matches manifest: True


Replay without any LLM:

In [4]:
report = replay(record.run_id, settings)
print(report.message, "| all match:", report.all_match)
pd.DataFrame([s.model_dump() for s in report.steps])

7 of 7 step outputs match; replay status completed, original status completed. | all match: True


,step_id,capability,capability_code,original_hash,replay_hash,match
0,s1,load_series,C01,19533e0349f1c64363fb6ca588288acb676fe8f3b3baba27df2a9de674466700,19533e0349f1c64363fb6ca588288acb676fe8f3b3baba27df2a9de674466700,True
1,s2,resample,C02,0c44f38c8ce12abc086f1416f5bd5a174afe0cfa3b6b38d52efcfad89d19489f,0c44f38c8ce12abc086f1416f5bd5a174afe0cfa3b6b38d52efcfad89d19489f,True
2,s3,compute_returns,C03,b607a740b7872e1349e5223ab3b22513417c6949a7949ad05bc222e3430d5e04,b607a740b7872e1349e5223ab3b22513417c6949a7949ad05bc222e3430d5e04,True
3,s4,flag_regime,C07,f7a9b4c68ffd2f92fa342bc29422e9382514277481ee46ee93e8368b34b5bbfe,f7a9b4c68ffd2f92fa342bc29422e9382514277481ee46ee93e8368b34b5bbfe,True
4,s5,forward_return,C08,b4b3370e2ab07a1ba6cc4d825785a3ae3e2441fecaafa9f157b7b345896af1f3,b4b3370e2ab07a1ba6cc4d825785a3ae3e2441fecaafa9f157b7b345896af1f3,True
5,s6,compare_groups,C09,cbc0b92cfc30cc2615925a69b333023f2c652fbdec336f280bb9365b40f9de5a,cbc0b92cfc30cc2615925a69b333023f2c652fbdec336f280bb9365b40f9de5a,True
6,s7,hac_mean_difference,C10,aeb85fa1b37303b494f2796d575b9007d1c3c188b4169f305748e55c8f6e648e,aeb85fa1b37303b494f2796d575b9007d1c3c188b4169f305748e55c8f6e648e,True


Traceability: trace the significance test (step s7, where the p-value comes from) back to
the NASDAQ file, through both of its inputs:

In [5]:
lineage = trace(record, "s7")
pd.DataFrame(
    [
        {
            "step": s.step_id,
            "code": s.capability_code,
            "capability": s.capability,
            "inputs": s.inputs,
            "params": s.params,
            "output sha256": s.output_hash[:12],
        }
        for s in lineage.chain
    ]
)

,step,code,capability,inputs,params,output sha256
0,s7,C10,hac_mean_difference,"{'outcome': 's5', 'flags': 's4'}",{},aeb85fa1b373
1,s5,C08,forward_return,{'series': 's2'},{'method': 'simple'},b4b3370e2ab0
2,s4,C07,flag_regime,{'series': 's3'},"{'quantile': 0.1, 'method': 'expanding', 'min_periods': 52, 'direction': 'below'}",f7a9b4c68ffd
3,s3,C03,compute_returns,{'series': 's2'},"{'method': 'simple', 'periods': 1}",b607a740b787
4,s2,C02,resample,{'series': 's1'},"{'freq': 'W', 'how': 'last'}",0c44f38c8ce1
5,s1,C01,load_series,{},{'dataset_id': 'NASDAQCOM'},19533e0349f1


In [6]:
pd.DataFrame([s.model_dump() for s in lineage.sources])

,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,NASDAQCOM,NASDAQCOM,b1c65ade24861ff5dc51bd5bbd4d8b7ed8e68af20a6181b2457c6bfe64289520,2026-10-17T09:22:36Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=NASDAQCOM,14042,1971-02-05,2026-10-15
